# Wav2Vec2 ASR Training for Twi
## Self-Supervised Speech Recognition Model

This notebook fine-tunes Wav2Vec2 models specifically for Twi language ASR, including the Akan-specific model.

In [ ]:
# Installation and imports
# Fix numpy version conflicts first
!pip install -q numpy>=2.0.0
!pip install -q transformers datasets torch torchaudio evaluate jiwer tensorboard
!pip install -q accelerate librosa soundfile pyctcdecode
# Install kenlm separately to avoid conflicts
!pip install -q https://github.com/kpu/kenlm/archive/master.zip

import os
import sys
import json
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Audio processing
import torchaudio
import librosa
import soundfile as sf

# Wav2Vec2 imports
from transformers import (
    Wav2Vec2ForCTC,
    Wav2Vec2Processor,
    Wav2Vec2ProcessorWithLM,
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset, DatasetDict, Audio
import evaluate

# Language model decoding
try:
    from pyctcdecode import build_ctcdecoder
    PYCTCDECODE_AVAILABLE = True
except ImportError:
    PYCTCDECODE_AVAILABLE = False
    print("⚠️ pyctcdecode not available, language model decoding disabled")

# Check environment
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'kaggle_secrets' in sys.modules

print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🎯 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"📊 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

print(f"\n🌍 Environment: {'Colab' if IN_COLAB else 'Kaggle' if IN_KAGGLE else 'Local'}")

# Auto-detect and use correct data path
if IN_COLAB:
    # Mount Google Drive if in Colab
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_PATH = Path('/content/drive/MyDrive/voice-lab-data')
elif IN_KAGGLE:
    DATA_PATH = Path('/kaggle/working/voice-lab-data')
else:
    # Local environment
    DATA_PATH = Path('/opt/kasaai/apps/voice-lab/data')

print(f"📂 Data path: {DATA_PATH}")

# Check if processed data exists
metadata_path = DATA_PATH / 'metadata/all_datasets_processed.csv'
if metadata_path.exists():
    processed_data = pd.read_csv(metadata_path)
    print(f"✅ Found processed data: {len(processed_data)} samples")
    print(f"   Languages: {processed_data['language'].unique() if 'language' in processed_data.columns else 'N/A'}")
    print(f"   Datasets: {processed_data['dataset'].unique() if 'dataset' in processed_data.columns else 'N/A'}")
else:
    print(f"⚠️ Processed data not found at {metadata_path}")
    print("   Please run notebooks 001 and 002 first to download and process data")

In [ ]:
# Configuration
@dataclass
class Wav2Vec2Config:
    """Configuration for Wav2Vec2 training."""
    # Model settings - Multiple options
    base_model: str = "facebook/wav2vec2-large-xlsr-53"  # Multilingual base
    akan_model: str = "azunre/wav2vec2-akan"  # Akan-specific model
    use_akan_model: bool = True  # Use Akan model if available
    
    # Data paths - use global DATA_PATH
    data_dir: Path = DATA_PATH
    output_dir: Path = DATA_PATH.parent / "models" / "wav2vec2_twi"
    cache_dir: Path = DATA_PATH.parent / "cache"
    
    # Training parameters
    batch_size: int = 16
    gradient_accumulation_steps: int = 2
    learning_rate: float = 1e-4
    weight_decay: float = 0.005
    warmup_steps: int = 1000
    num_train_epochs: int = 30
    fp16: bool = True
    gradient_checkpointing: bool = True
    
    # Audio settings
    sample_rate: int = 16000
    max_duration: float = 20.0
    
    # Wav2Vec2 specific
    freeze_feature_encoder: bool = True
    mask_time_prob: float = 0.05
    mask_feature_prob: float = 0.0
    ctc_loss_reduction: str = "mean"
    ctc_zero_infinity: bool = False
    
    # Language model
    use_language_model: bool = True
    lm_path: Optional[Path] = None
    
    # Evaluation
    eval_steps: int = 500
    save_steps: int = 500
    logging_steps: int = 100
    
    def __post_init__(self):
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.cache_dir.mkdir(parents=True, exist_ok=True)

config = Wav2Vec2Config()
print(f"📋 Wav2Vec2 Configuration loaded")
print(f"🎯 Using model: {config.akan_model if config.use_akan_model else config.base_model}")
print(f"   Data directory: {config.data_dir}")
print(f"   Output directory: {config.output_dir}")

In [ ]:
# Twi vocabulary and tokenizer setup
class TwiVocabularyBuilder:
    """Build Twi-specific vocabulary for Wav2Vec2."""
    
    @staticmethod
    def get_twi_vocab() -> Dict[str, int]:
        """Create Twi vocabulary including special characters."""
        # Base characters
        chars = list("abcdefghijklmnopqrstuvwxyz")
        
        # Twi-specific characters
        twi_chars = ['ɛ', 'ɔ']
        chars.extend(twi_chars)
        
        # Add space and apostrophe
        chars.extend([' ', "'"])
        
        # Create vocabulary with special tokens
        vocab = {"[PAD]": 0, "[UNK]": 1, "[CTC]": 2}
        
        # Add characters
        for i, char in enumerate(chars):
            vocab[char] = i + 3
        
        # Add uppercase versions
        for char in list("ABCDEFGHIJKLMNOPQRSTUVWXYZ") + ['Ɛ', 'Ɔ']:
            if char.lower() in vocab:
                vocab[char] = len(vocab)
        
        return vocab
    
    @staticmethod
    def extract_vocab_from_data(texts: List[str]) -> Dict[str, int]:
        """Extract vocabulary from actual Twi texts."""
        all_chars = set()
        
        for text in texts:
            all_chars.update(text.lower())
        
        # Create vocabulary
        vocab = {"[PAD]": 0, "[UNK]": 1, "[CTC]": 2}
        
        for i, char in enumerate(sorted(all_chars)):
            vocab[char] = i + 3
        
        return vocab
    
    @staticmethod
    def save_vocab(vocab: Dict[str, int], path: Path):
        """Save vocabulary to file."""
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(vocab, f, ensure_ascii=False, indent=2)
        print(f"💾 Vocabulary saved to {path}")

# Build Twi vocabulary
vocab_builder = TwiVocabularyBuilder()
twi_vocab = vocab_builder.get_twi_vocab()
print(f"🔤 Twi vocabulary size: {len(twi_vocab)}")
print(f"📝 Sample vocabulary: {list(twi_vocab.items())[:10]}")

In [ ]:
# Model and processor initialization
class Wav2Vec2ModelLoader:
    """Load and configure Wav2Vec2 models."""
    
    def __init__(self, config: Wav2Vec2Config, vocab: Dict[str, int]):
        self.config = config
        self.vocab = vocab
        self.model = None
        self.processor = None
        
    def load_akan_model(self):
        """Load pre-trained Akan Wav2Vec2 model."""
        try:
            print(f"🌍 Loading Akan-specific model: {self.config.akan_model}")
            
            self.processor = Wav2Vec2Processor.from_pretrained(
                self.config.akan_model,
                cache_dir=self.config.cache_dir
            )
            
            self.model = Wav2Vec2ForCTC.from_pretrained(
                self.config.akan_model,
                cache_dir=self.config.cache_dir,
                ctc_loss_reduction=self.config.ctc_loss_reduction,
                pad_token_id=self.processor.tokenizer.pad_token_id,
                vocab_size=len(self.processor.tokenizer)
            )
            
            print("✅ Akan model loaded successfully")
            return True
            
        except Exception as e:
            print(f"⚠️ Could not load Akan model: {e}")
            return False
    
    def load_base_model(self):
        """Load base multilingual Wav2Vec2 model."""
        print(f"🌐 Loading base model: {self.config.base_model}")
        
        # Create tokenizer with Twi vocabulary
        tokenizer = Wav2Vec2CTCTokenizer(
            vocab_file=None,
            vocab_dict=self.vocab,
            unk_token="[UNK]",
            pad_token="[PAD]",
            word_delimiter_token=" ",
        )
        
        # Create feature extractor
        feature_extractor = Wav2Vec2FeatureExtractor(
            feature_size=1,
            sampling_rate=self.config.sample_rate,
            padding_value=0.0,
            do_normalize=True,
            return_attention_mask=True,
        )
        
        # Create processor
        self.processor = Wav2Vec2Processor(
            feature_extractor=feature_extractor,
            tokenizer=tokenizer
        )
        
        # Load model
        self.model = Wav2Vec2ForCTC.from_pretrained(
            self.config.base_model,
            cache_dir=self.config.cache_dir,
            ctc_loss_reduction=self.config.ctc_loss_reduction,
            pad_token_id=self.processor.tokenizer.pad_token_id,
            vocab_size=len(self.vocab),
            ignore_mismatched_sizes=True
        )
        
        print("✅ Base model loaded and configured for Twi")
    
    def configure_model(self):
        """Configure model for training."""
        # Freeze feature encoder if specified
        if self.config.freeze_feature_encoder:
            self.model.freeze_feature_encoder()
            print("❄️ Feature encoder frozen")
        
        # Enable gradient checkpointing
        if self.config.gradient_checkpointing:
            self.model.gradient_checkpointing_enable()
            print("✅ Gradient checkpointing enabled")
        
        # Set training mode
        self.model.train()
        
        print(f"📊 Model parameters: {self.model.num_parameters():,}")
        trainable = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        print(f"🎯 Trainable parameters: {trainable:,}")
    
    def load(self):
        """Load model based on configuration."""
        if self.config.use_akan_model:
            if not self.load_akan_model():
                print("📥 Falling back to base model...")
                self.load_base_model()
        else:
            self.load_base_model()
        
        self.configure_model()
        return self.model, self.processor

# Load model
model_loader = Wav2Vec2ModelLoader(config, twi_vocab)
model, processor = model_loader.load()

In [ ]:
# Data loading and preprocessing
class Wav2Vec2DataProcessor:
    """Process data for Wav2Vec2 training."""
    
    def __init__(self, processor, config: Wav2Vec2Config):
        self.processor = processor
        self.config = config
        
    def load_data(self) -> DatasetDict:
        """Load Twi datasets."""
        all_data = []
        
        # Load from metadata
        metadata_path = self.config.data_dir / "metadata/all_datasets_processed.csv"
        if metadata_path.exists():
            print("📊 Loading processed data...")
            df = pd.read_csv(metadata_path)
            
            if 'audio_path' in df.columns and 'processed_text' in df.columns:
                df = df.dropna(subset=['audio_path', 'processed_text'])
                
                # Update audio paths to be absolute
                df['audio'] = df['audio_path'].apply(
                    lambda x: str(self.config.data_dir / 'audio' / Path(x).name) if pd.notna(x) else None
                )
                
                # Check file existence
                df['exists'] = df['audio'].apply(lambda x: Path(x).exists() if pd.notna(x) else False)
                existing_df = df[df['exists']]
                
                if len(existing_df) > 0:
                    all_data.append(existing_df[['audio', 'processed_text']].rename(columns={'processed_text': 'text'}))
                    print(f"✅ Loaded {len(existing_df)} samples from metadata")
                else:
                    print(f"⚠️ No audio files found at expected paths")
        
        if all_data:
            combined_df = pd.concat(all_data, ignore_index=True)
            dataset = Dataset.from_pandas(combined_df)
            return self._create_splits(dataset)
        else:
            print("⚠️ No real data found, creating dummy dataset for testing")
            return self._create_dummy_data()
    
    def _create_splits(self, dataset: Dataset) -> DatasetDict:
        """Create train/val/test splits."""
        dataset = dataset.shuffle(seed=42)
        
        train_size = int(0.8 * len(dataset))
        val_size = int(0.1 * len(dataset))
        
        return DatasetDict({
            'train': dataset.select(range(train_size)),
            'validation': dataset.select(range(train_size, train_size + val_size)),
            'test': dataset.select(range(train_size + val_size, len(dataset)))
        })
    
    def _create_dummy_data(self) -> DatasetDict:
        """Create dummy dataset."""
        # Create dummy audio files
        dummy_dir = self.config.data_dir / "dummy_audio"
        dummy_dir.mkdir(parents=True, exist_ok=True)
        
        dummy_data = []
        for i, text in enumerate([
            "Wo ho te sɛn",
            "Me ma wo akye",
            "Me da wo ase",
            "Ɛyɛ sɛn",
            "Nante yie"
        ]):
            # Create dummy audio file
            audio_file = dummy_dir / f"dummy_{i}.wav"
            if not audio_file.exists():
                # Generate 1 second of silence
                dummy_audio = np.zeros(16000, dtype=np.float32)
                sf.write(audio_file, dummy_audio, 16000)
            
            dummy_data.append({"text": text, "audio": str(audio_file)})
        
        dataset = Dataset.from_pandas(pd.DataFrame(dummy_data))
        return DatasetDict({'train': dataset, 'validation': dataset, 'test': dataset})
    
    def prepare_dataset(self, batch):
        """Prepare batch for training."""
        audio_arrays = []
        
        for audio_path in batch["audio"]:
            if Path(audio_path).exists():
                audio, sr = librosa.load(audio_path, sr=self.config.sample_rate)
                
                # Limit duration
                max_samples = int(self.config.max_duration * self.config.sample_rate)
                if len(audio) > max_samples:
                    audio = audio[:max_samples]
                
                audio_arrays.append(audio)
            else:
                audio_arrays.append(np.zeros(self.config.sample_rate, dtype=np.float32))
        
        # Process audio
        batch["input_values"] = self.processor(
            audio_arrays,
            sampling_rate=self.config.sample_rate,
            padding=True,
            return_tensors="pt"
        ).input_values
        
        # Process text
        with self.processor.as_target_processor():
            batch["labels"] = self.processor(
                batch["text"],
                padding=True,
                return_tensors="pt"
            ).input_ids
        
        return batch

# Load and prepare data
data_processor = Wav2Vec2DataProcessor(processor, config)
datasets = data_processor.load_data()

print(f"\n📊 Dataset Statistics:")
for split, dataset in datasets.items():
    print(f"  {split}: {len(dataset)} samples")

# Prepare datasets
print("\n🔄 Preparing datasets...")
prepared_datasets = datasets.map(
    data_processor.prepare_dataset,
    batched=True,
    batch_size=8,
    remove_columns=datasets["train"].column_names
)
print("✅ Data preparation complete!")

In [ ]:
# Data collator
from dataclasses import dataclass
from typing import Dict, List, Union
import torch

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True
    
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Separate inputs and labels
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        
        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )
        
        with self.processor.as_target_processor():
            labels_batch = self.processor.pad(
                label_features,
                padding=self.padding,
                return_tensors="pt",
            )
        
        # Replace padding with -100 for loss calculation
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        
        batch["labels"] = labels
        
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

In [ ]:
# Training
class Wav2Vec2Trainer:
    """Trainer for Wav2Vec2 models."""
    
    def __init__(self, model, processor, config: Wav2Vec2Config):
        self.model = model
        self.processor = processor
        self.config = config
        self.trainer = None
        
    def compute_metrics(self, eval_pred):
        """Compute WER and CER metrics."""
        pred_logits = eval_pred.predictions
        pred_ids = np.argmax(pred_logits, axis=-1)
        
        # Replace -100 with pad token
        eval_pred.label_ids[eval_pred.label_ids == -100] = self.processor.tokenizer.pad_token_id
        
        # Decode
        pred_str = self.processor.batch_decode(pred_ids)
        label_str = self.processor.batch_decode(eval_pred.label_ids, group_tokens=False)
        
        # Calculate metrics
        wer = evaluate.load("wer")
        cer = evaluate.load("cer")
        
        wer_score = wer.compute(predictions=pred_str, references=label_str)
        cer_score = cer.compute(predictions=pred_str, references=label_str)
        
        return {"wer": wer_score, "cer": cer_score}
    
    def setup_training(self, train_dataset, eval_dataset, data_collator):
        """Setup training configuration."""
        
        training_args = TrainingArguments(
            output_dir=str(self.config.output_dir),
            group_by_length=True,
            per_device_train_batch_size=self.config.batch_size,
            per_device_eval_batch_size=self.config.batch_size,
            gradient_accumulation_steps=self.config.gradient_accumulation_steps,
            evaluation_strategy="steps",
            save_strategy="steps",
            eval_steps=self.config.eval_steps,
            save_steps=self.config.save_steps,
            logging_steps=self.config.logging_steps,
            learning_rate=self.config.learning_rate,
            weight_decay=self.config.weight_decay,
            warmup_steps=self.config.warmup_steps,
            num_train_epochs=self.config.num_train_epochs,
            save_total_limit=2,
            load_best_model_at_end=True,
            metric_for_best_model="wer",
            greater_is_better=False,
            fp16=self.config.fp16,
            gradient_checkpointing=self.config.gradient_checkpointing,
            push_to_hub=False,
            report_to=["tensorboard"],
            remove_unused_columns=False,
        )
        
        self.trainer = Trainer(
            model=self.model,
            data_collator=data_collator,
            args=training_args,
            compute_metrics=self.compute_metrics,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            tokenizer=self.processor.feature_extractor,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
        )
        
        print("✅ Trainer initialized!")
    
    def train(self):
        """Start training."""
        print(f"\n🏋️ Starting Wav2Vec2 training...")
        print(f"📊 Training on {torch.cuda.device_count()} GPU(s)")
        
        # Train
        train_result = self.trainer.train()
        
        # Save
        self.trainer.save_model()
        self.processor.save_pretrained(self.config.output_dir)
        
        # Save metrics
        with open(self.config.output_dir / "training_results.json", "w") as f:
            json.dump(train_result.metrics, f, indent=2)
        
        print("✅ Training complete!")
        return train_result

# Setup training
wav2vec2_trainer = Wav2Vec2Trainer(model, processor, config)
wav2vec2_trainer.setup_training(
    prepared_datasets["train"],
    prepared_datasets["validation"],
    data_collator
)

# Uncomment to start training
# train_result = wav2vec2_trainer.train()

In [ ]:
# Language Model Integration
class LanguageModelDecoder:
    """Add language model decoding to Wav2Vec2."""
    
    def __init__(self, processor, config: Wav2Vec2Config):
        self.processor = processor
        self.config = config
        self.decoder = None
        
    def build_lm_from_texts(self, texts: List[str], save_path: Path):
        """Build n-gram language model from Twi texts."""
        if not PYCTCDECODE_AVAILABLE:
            print("⚠️ pyctcdecode not available, skipping language model")
            return None
            
        print("🔨 Building language model from texts...")
        
        # Prepare text file for KenLM
        lm_text_path = save_path / "lm_texts.txt"
        with open(lm_text_path, 'w', encoding='utf-8') as f:
            for text in texts:
                f.write(text.lower() + "\n")
        
        # Build ARPA file using KenLM
        import subprocess
        arpa_path = save_path / "twi_lm.arpa"
        
        try:
            # Build language model
            subprocess.run([
                "lmplz", "-o", "3",
                "--text", str(lm_text_path),
                "--arpa", str(arpa_path)
            ], check=True)
            
            # Convert to binary
            binary_path = save_path / "twi_lm.bin"
            subprocess.run([
                "build_binary",
                str(arpa_path),
                str(binary_path)
            ], check=True)
            
            print(f"✅ Language model saved to {binary_path}")
            return binary_path
            
        except Exception as e:
            print(f"⚠️ Could not build language model: {e}")
            print("   Install KenLM: pip install https://github.com/kpu/kenlm/archive/master.zip")
            return None
    
    def create_decoder_with_lm(self, lm_path: Path):
        """Create CTC decoder with language model."""
        if not PYCTCDECODE_AVAILABLE:
            print("⚠️ pyctcdecode not available")
            return None
            
        if not lm_path or not lm_path.exists():
            print("⚠️ No language model found")
            return None
        
        # Get vocabulary
        vocab_dict = self.processor.tokenizer.get_vocab()
        sorted_vocab = sorted((k, v) for k, v in vocab_dict.items() if v != self.processor.tokenizer.unk_token_id)
        vocab_list = [k for k, _ in sorted_vocab]
        
        # Build decoder
        self.decoder = build_ctcdecoder(
            labels=vocab_list,
            kenlm_model_path=str(lm_path),
            alpha=0.5,  # LM weight
            beta=1.0,   # Word bonus
        )
        
        print("✅ Decoder with language model created")
        return self.decoder
    
    def decode_with_lm(self, logits: np.ndarray) -> str:
        """Decode using language model."""
        if self.decoder is None:
            # Fallback to greedy decoding
            pred_ids = np.argmax(logits, axis=-1)
            return self.processor.decode(pred_ids)
        
        # Use beam search with LM
        return self.decoder.decode(logits)

# Build language model (if texts available)
# lm_builder = LanguageModelDecoder(processor, config)
# lm_path = lm_builder.build_lm_from_texts(train_texts, config.output_dir)
# decoder = lm_builder.create_decoder_with_lm(lm_path)

In [ ]:
# Evaluation
class Wav2Vec2Evaluator:
    """Evaluate Wav2Vec2 model."""
    
    def __init__(self, model_path: Path, test_dataset, processor):
        self.model_path = model_path
        self.test_dataset = test_dataset
        self.processor = processor
        self.model = None
        
    def load_model(self):
        """Load trained model."""
        self.model = Wav2Vec2ForCTC.from_pretrained(self.model_path)
        self.model.eval()
        
        if torch.cuda.is_available():
            self.model = self.model.cuda()
        
        print(f"✅ Model loaded from {self.model_path}")
    
    def evaluate(self, use_lm: bool = False, decoder=None) -> Dict:
        """Evaluate model performance."""
        print(f"\n📊 Evaluating Wav2Vec2 model {'with LM' if use_lm else 'without LM'}...")
        
        predictions = []
        references = []
        inference_times = []
        
        for item in tqdm(self.test_dataset):
            # Load audio
            if Path(item["audio"]).exists():
                audio, _ = librosa.load(item["audio"], sr=16000)
            else:
                audio = np.zeros(16000, dtype=np.float32)
            
            # Process
            inputs = self.processor(
                audio,
                sampling_rate=16000,
                return_tensors="pt",
                padding=True
            )
            
            if torch.cuda.is_available():
                inputs = {k: v.cuda() for k, v in inputs.items()}
            
            # Inference
            import time
            start = time.time()
            
            with torch.no_grad():
                logits = self.model(**inputs).logits
            
            inference_time = time.time() - start
            inference_times.append(inference_time)
            
            # Decode
            if use_lm and decoder:
                # Use language model
                logits_np = logits.cpu().numpy()[0]
                transcription = decoder.decode(logits_np)
            else:
                # Greedy decoding
                predicted_ids = torch.argmax(logits, dim=-1)
                transcription = self.processor.batch_decode(predicted_ids)[0]
            
            predictions.append(transcription)
            references.append(item["text"])
        
        # Calculate metrics
        wer_metric = evaluate.load("wer")
        cer_metric = evaluate.load("cer")
        
        wer = wer_metric.compute(predictions=predictions, references=references)
        cer = cer_metric.compute(predictions=predictions, references=references)
        
        results = {
            "wer": wer,
            "cer": cer,
            "avg_inference_time": np.mean(inference_times),
            "total_samples": len(predictions),
            "with_lm": use_lm,
            "sample_predictions": list(zip(references[:5], predictions[:5]))
        }
        
        print(f"\n📊 Results:")
        print(f"  WER: {wer:.2%}")
        print(f"  CER: {cer:.2%}")
        print(f"  Avg time: {np.mean(inference_times):.3f}s")
        
        return results

# Evaluate model
# evaluator = Wav2Vec2Evaluator(config.output_dir, datasets["test"], processor)
# evaluator.load_model()
# results_no_lm = evaluator.evaluate(use_lm=False)
# results_with_lm = evaluator.evaluate(use_lm=True, decoder=decoder)

## Summary

This notebook implements Wav2Vec2 for Twi ASR with:

### Key Features:
1. **Pre-trained Models**: Uses XLSR-53 multilingual or Akan-specific model
2. **Custom Vocabulary**: Twi-specific character set including ɛ, ɔ
3. **CTC Loss**: Connectionist Temporal Classification for alignment
4. **Language Model**: Optional n-gram LM for better decoding
5. **Efficient Training**: Feature encoder freezing, gradient checkpointing

### Model Options:
- **facebook/wav2vec2-large-xlsr-53**: 300M param multilingual model
- **azunre/wav2vec2-akan**: Pre-trained on Akan languages

### Advantages:
- Self-supervised pre-training
- Works well with limited labeled data
- Language model integration
- Good for low-resource languages

### Next Steps:
1. Train on full Twi dataset
2. Build Twi-specific language model
3. Compare with Whisper and MMS
4. Optimize for edge deployment